<a href="https://colab.research.google.com/github/Viren1212/Makhana-Work/blob/main/colab_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Import essential libraries grouped by functionality for clarity
# Image Processing and Computer Vision
import cv2                  # OpenCV for core image processing and computer vision operations
import numpy as np          # NumPy for efficient numerical computations and array manipulations
from sklearn.cluster import KMeans  # For ordering image using Kmeans



# Data Analysis and Visualization
import matplotlib.pyplot as plt  # Matplotlib for creating plots and visualizations
import pandas as pd         # Pandas for structured data handling and statistical analysis
import seaborn as sns       # Seaborn for advanced statistical visualizations (e.g., heatmaps)
from matplotlib.backends.backend_pdf import PdfPages  # Enables PDF report generation
from matplotlib.gridspec import GridSpec  # Advanced layout control for Matplotlib figures
import numpy as np

# Utilities and File Management
import math                 # Mathematical functions (e.g., sqrt, pi) for metric calculations
import os                   # Operating system utilities for file and directory operations
import json                 # JSON handling for calibration and metrics caching
import logging              # Logging for tracking events, errors, and process flow
from google.colab import files  # Google Colab-specific utility for file uploads/downloads
from collections import Counter  # Efficient counting of grade occurrences
from datetime import datetime  # Timestamp generation for unique identifiers and metadata
import pytz
IST = pytz.timezone('Asia/Kolkata')
def get_ist_now():
    return datetime.now(pytz.utc).astimezone(IST)


# Parallel Processing
import concurrent.futures   # Framework for concurrent task execution
from concurrent.futures import ProcessPoolExecutor, as_completed  # Tools for process-based parallelism


# Matplotlib Compatibility Handling
import matplotlib           # Base Matplotlib module for version checks
# Ensure colormap compatibility across Matplotlib versions
if hasattr(matplotlib, 'colormaps'):
    get_cmap = lambda name: matplotlib.colormaps[name]
else:
    from matplotlib.cm import get_cmap



In [2]:
# ===========================================================================
# Enhanced Fixed Setup Calibration System
# Purpose: Convert pixel measurements to real-world units (mm) for accurate grading
# ===========================================================================


# File path for storing calibration data
CALIBRATION_FILE = "enhanced_calibration.json"
SOFTWARE_VERSION = "v1.0"


# Schema to validate calibration data structure
CALIBRATION_SCHEMA = {
    "required": ["mm_per_pixel", "calibration_date"],  # Essential fields for all calibration methods
    "optional": ["camera_height_mm", "reference_diameter_mm", "method", "sensor_width", "focal_length"]  # Optional fields based on method
}


def validate_calibration_data(data):
    """
    Validates calibration data against the schema to ensure integrity.


    Args:
        data (dict): Calibration data dictionary to check.


    Returns:
        bool: True if data is valid.


    Raises:
        ValueError: If required fields are missing or geometric method lacks camera height.
    """
    for field in CALIBRATION_SCHEMA["required"]:
        if field not in data:
            raise ValueError(f"Missing required field: {field}")
    if data.get("method", "reference") == "geometric" and "camera_height_mm" not in data:
        raise ValueError("Missing required field 'camera_height_mm' for geometric calibration")
    return True


def calibrate_with_reference():
    """
    Calibrates pixel-to-mm conversion using a reference object (e.g., coin) in an image.


    Returns:
        float: Calibration factor (mm_per_pixel).


    Raises:
        FileNotFoundError: If reference image fails to load.
        ValueError: If no circles are detected.
    """
    logging.info("\n=== Starting Reference Object Calibration ===")
    uploaded = files.upload()  # Prompt user to upload an image with a reference object
    ref_path = list(uploaded.keys())[0]  # Extract the uploaded file path
    ref_image = cv2.imread(ref_path)  # Load the image
    if ref_image is None:
        logging.error("Failed to load reference image")
        raise FileNotFoundError("Failed to load reference image")
    gray_ref = cv2.cvtColor(ref_image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale for processing
    # Detect circles using Hough Transform with tuned parameters
    circles = cv2.HoughCircles(
        gray_ref, cv2.HOUGH_GRADIENT, dp=1, minDist=100,
        param1=50, param2=30, minRadius=10, maxRadius=100
    )
    if circles is None or len(circles) == 0:
        logging.error("No circles detected in reference image")
        raise ValueError("No circles detected in reference image")
    circles = circles[0, :]  # Flatten circle data
    ref_circle = max(circles, key=lambda x: x[2])
    px_diameter = 2 * ref_circle[2]
    ref_object_name = input("Enter reference object name (e.g., Indian ₹10 Coin): ")
    actual_mm = float(input("Enter reference object diameter (mm): "))
    mm_per_pixel = actual_mm / px_diameter
    image_height, image_width = ref_image.shape[:2]
    logging.info(f"Calculated mm_per_pixel: {mm_per_pixel:.4f}")

    ref_image_path = "calibration_reference_image.png"
    cv2.imwrite(ref_image_path, ref_image)

    return {
        'mm_per_pixel': mm_per_pixel,
        'reference_object': ref_object_name,
        'actual_diameter_mm': actual_mm,
        'measured_diameter_px': round(px_diameter, 2),
        'image_resolution': f"{image_width} x {image_height}",
        'reference_image_path': ref_image_path
    }


def geometric_calibration():
    """
    Calibrates using camera geometry specifications provided by the user.


    Returns:
        tuple: (mm_per_pixel, camera_height_mm) calibration factor and height.


    Notes:
        Requires precise user inputs for accurate results.
    """
    logging.info("\n=== Starting Geometric Calibration ===")
    specs = {}  # Dictionary to store camera specifications
    # Prompt user for required camera parameters
    for param in [('sensor_width_mm', "sensor width (mm)"), ('focal_length_mm', "focal length (mm)"), ('image_width_px', "image width (pixels)")]:
        specs[param[0]] = float(input(f"Enter {param[1]}: "))
    camera_height = float(input("Enter camera height from base (mm): "))
    # Calculate mm_per_pixel using geometric formula
    mm_per_pixel = (specs['sensor_width_mm'] * camera_height) / (specs['focal_length_mm'] * specs['image_width_px'])
    logging.info(f"Calculated mm_per_pixel: {mm_per_pixel:.4f} with camera height: {camera_height} mm")
    return mm_per_pixel, camera_height


def save_calibration(data):
    """
    Saves calibration data to a JSON file for persistence.


    Args:
        data (dict): Calibration data to save.
    """
    # Convert NumPy types to native Python types for JSON compatibility
    for key, value in data.items():
        if isinstance(value, (np.generic,)):
            data[key] = value.item()
    # Add metadata for tracking
    data.update({
        "version": "1.1",
        "calibration_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })
    logging.info(f"Saving calibration: method={data['method']}, mm_per_pixel={data['mm_per_pixel']:.4f}")
    with open(CALIBRATION_FILE, 'w') as f:
        json.dump(data, f, indent=2)  # Write with indentation for readability
    logging.info(f"Calibration data saved to {CALIBRATION_FILE}")


def load_calibration():
    """
    Loads existing calibration data from file if it exists and is valid.


    Returns:
        dict or None: Calibration data if loaded successfully, None otherwise.
    """
    try:
        with open(CALIBRATION_FILE, 'r') as f:
            data = json.load(f)
        validate_calibration_data(data)  # Ensure data meets schema requirements
        logging.info(f"Loaded calibration: method={data['method']}, mm_per_pixel={data['mm_per_pixel']:.4f}, date={data['calibration_date']}")
        return data
    except (IOError, ValueError, KeyError) as e:
        logging.warning(f"Calibration loading failed: {str(e)}")
        return None


def calibration_menu():
    """
    Interactive menu to select and execute a calibration method.


    Returns:
        dict: Calibration data including method and mm_per_pixel.


    Raises:
        ValueError: If an invalid method is selected.
    """
    logging.info("\n=== Calibration Method Selection ===")
    print("1. Reference Object (Recommended)\n2. Geometric (Camera Specs)\n3. Manual Entry")
    choice = input("Select method (1-3): ")
    camera_model = input("Enter camera model (e.g., JAI AD-130GE): ")
    if choice == '1':
        ref_data = calibrate_with_reference()
        return {'method': 'reference', 'camera_model': camera_model, **ref_data}
    elif choice == '2':
        mm_px, cam_height = geometric_calibration()
        return {'method': 'geometric', 'camera_model': camera_model, 'mm_per_pixel': mm_px, 'camera_height_mm': cam_height}
    elif choice == '3':
        mm_per_pixel = float(input("Enter known mm/pixel value: "))
        return {'method': 'manual', 'camera_model': camera_model, 'mm_per_pixel': mm_per_pixel}
    logging.error("Invalid calibration method selected")
    raise ValueError("Invalid selection")

def get_calibration_factor(recalibrate=True):
    """
    Retrieves or computes the calibration factor based on user preference.


    Args:
        recalibrate (bool): If True, forces new calibration; if False, tries to load existing data.


    Returns:
        float: Calibration factor (mm_per_pixel).
    """
def get_calibration_factor(recalibrate=True):
    if not recalibrate:
        data = load_calibration()
        if data:
            logging.info(f"Using loaded {data['method']} calibration from {data['calibration_date']}")
            return data['mm_per_pixel'], data
    logging.info("No valid calibration found or recalibration requested. Starting new calibration...")
    calib_data = calibration_menu()
    calib_data['calibration_date'] = get_ist_now().strftime("%Y-%m-%d")
    calib_data['calibration_status'] = 'Successful'
    save_calibration(calib_data)
    return calib_data['mm_per_pixel'], calib_data

In [3]:
# ===========================================================================
# Configurable Thresholds
# Purpose: Define criteria for filtering and grading makhanas
# ===========================================================================


# Thresholds for processing and grading makhanas
THRESHOLDS = {
    'diameter_mm': 40.0,          # Maximum allowed diameter in mm for grading
    'circularity': 0.3,           # Minimum circularity score to filter objects
    'aspect_ratio': 0.3,          # Minimum aspect ratio to consider valid objects
    'min_contour_area': 1000,     # Minimum contour area (pixels) to exclude noise
    'histogram_bins': 20,         # Number of bins for histogram-based color analysis
    'min_white_percentage': 50.0, # Minimum white percentage for quality grading
    'min_whiteness': 25.0         # Minimum whiteness (L* value) for grading
}


In [4]:
# ===========================================================================
# Batch Grading System with Weighted Scoring
# Purpose: Assign grades based on size, shape, and color metrics
# ===========================================================================


# Grade Driteria for makhanas based on multiple metrics
BATCH_GRADES = {
    'Grade A': {
        'min_diameter': 25, 'max_diameter': 60,
        'min_circularity': 0.9, 'max_circularity': 1.0,
        'min_aspect_ratio': 0.9, 'max_aspect_ratio': 1.0,
        'min_white_percentage': 90.0, 'max_white_percentage': 100.0,
        'min_whiteness': 50.0, 'max_whiteness': 100.0
    },
    'Grade B': {
        'min_diameter': 20, 'max_diameter': 25,
        'min_circularity': 0.8, 'max_circularity': 0.9,
        'min_aspect_ratio': 0.7, 'max_aspect_ratio': 0.9,
        'min_white_percentage': 80.0, 'max_white_percentage': 90.0,
        'min_whiteness': 40.0, 'max_whiteness': 50.0
    },
    'Grade C': {
        'min_diameter': 15, 'max_diameter': 20,
        'min_circularity': 0.6, 'max_circularity': 0.8,
        'min_aspect_ratio': 0.5, 'max_aspect_ratio': 0.7,
        'min_white_percentage': 70.0, 'max_white_percentage': 80.0,
        'min_whiteness': 35.0, 'max_whiteness': 40.0
    },
    'Grade D': {
        'min_diameter': 10, 'max_diameter': 15,
        'min_circularity': 0.4, 'max_circularity': 0.6,
        'min_aspect_ratio': 0.3, 'max_aspect_ratio': 0.5,
        'min_white_percentage': 60, 'max_white_percentage': 70.0,
        'min_whiteness': 20.0, 'max_whiteness': 35.0
    }
}


# Weights for metrics in the grading system (sum to 1.0 for balanced scoring)
METRIC_WEIGHTS = {
    'Effective Diameter (mm)': 0.25,
    'Circularity': 0.25,
    'Aspect Ratio': 0.1,
    'White Percentage': 0.2,
    'Whiteness': 0.2
}


# Mapping to compare each Grade Bgainst the next higher grade
COMPARISON_GRADE = {
    'Grade B': 'Grade A',
    'Grade C': 'Grade B',
    'Grade D': 'Grade C'
}


def normalize_metric(value, min_val, max_val):
    """
    Normalizes a metric value to a 0-1 range for weighted scoring.


    Args:
        value (float): Metric value to normalize.
        min_val (float): Minimum acceptable value.
        max_val (float): Maximum acceptable value.


    Returns:
        float: Normalized score between 0 and 1.
    """
    if max_val == min_val:  # Avoid division by zero
        return 1.0 if value >= min_val else 0.0
    return max(0.0, min(1.0, (value - min_val) / (max_val - min_val)))


def determine_grade(item_metrics):
    """
    Assigns a Grade Cased on weighted scoring of item metrics.


    Args:
        item_metrics (dict): Metrics for a single makhana (diameter, circularity, etc.).


    Returns:
        str: Assigned grade ('Grade A', 'Grade B', 'Grade C', 'Grade D').
    """
    for grade in ['Grade A', 'Grade B', 'Grade C']:
        specs = BATCH_GRADES[grade]
        total_score = 0.0
        for metric, weight in METRIC_WEIGHTS.items():
            base_key = 'diameter' if metric == 'Effective Diameter (mm)' else metric.lower().replace(' ', '_')
            min_key = f'min_{base_key}'
            max_key = f'max_{base_key}'
            min_val = specs[min_key]
            max_val = specs[max_key]
            value = item_metrics[metric]
            normalized = normalize_metric(value, min_val, max_val)
            total_score += weight * normalized
        if total_score >= 0.8:  # Threshold for Grade Bssignment
            logging.debug(f"Assigned grade {grade} with score {total_score:.2f}")
            return grade
    logging.debug("Assigned grade 'Grade D' (score below threshold)")
    return "Grade D"


def calculate_grade_distribution(metrics):
    """
    Computes the distribution of grades across a batch.


    Args:
        metrics (list): List of metric dictionaries for all makhanas.


    Returns:
        dict: Grade distribution with counts and percentages.
    """
    grade_counts = {grade: 0 for grade in BATCH_GRADES}
    for m in metrics:
        grade = m['Grade']
        grade_counts[grade] += 1
    total = len(metrics)
    dist = {grade: {'count': count, 'percentage': (count / total * 100) if total > 0 else 0}
            for grade, count in grade_counts.items()}
    logging.info(f"Grade distribution: {dist}")
    return dist

In [5]:
import numpy as np
# ===========================================================================
# Color Analysis Functions
# Purpose: Evaluate color properties (whiteness) of makhanas
# ===========================================================================


def analyze_color(image, mask):
    """
    Analyzes color properties of a segmented makhana to assess whiteness.


    Args:
        image (numpy.ndarray): Input image in BGR format.
        mask (numpy.ndarray): Binary mask of the makhana.


    Returns:
        tuple: (white_percentage, whiteness, avg_b_channel) - adjusted white area percentage, average lightness, and average b_channel.
    """
    try:
        # Convert to Lab color space for accurate lightness (L) measurement
        lab_image = cv2.cvtColor(image, cv2.COLOR_BGR2Lab)
        l_channel = lab_image[:, :, 0]  # Extract L channel (lightness)
        b_channel = lab_image[:, :, 2]  # Extract B channel (color) Line added
        # Erode mask to exclude edge artifacts
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        core_mask = cv2.erode(mask, kernel, iterations=1)
        # Apply mask to L channel
        masked_l = cv2.bitwise_and(l_channel, l_channel, mask=core_mask)
        # Identify dark spots (L < 70)
        dark_spot_mask = ((masked_l < 70) & (core_mask > 0)).astype(np.uint8) * 255
        # Clean noise in dark spot detection
        dark_clean = cv2.morphologyEx(dark_spot_mask, cv2.MORPH_OPEN, kernel)
        dark_clean = cv2.morphologyEx(dark_clean, cv2.MORPH_CLOSE, kernel)
        # Ignore tiny dark areas (< 0.2% of total area)
        dark_ratio = cv2.countNonZero(dark_clean) / (cv2.countNonZero(core_mask) + 1e-5)
        if dark_ratio < 0.002:
            dark_clean[:] = 0
        # Define white region by excluding dark spots
        white_region_mask = cv2.bitwise_and(core_mask, cv2.bitwise_not(dark_clean))
        # # Calculate whiteness as mean L value of white region
        white_region_pixels = masked_l[white_region_mask > 0]
        # whiteness = np.mean(white_region_pixels) if white_region_pixels.size > 0 else 0

        # Calculate average b_channel for the white region
        b_channel_white_region_pixels = b_channel[white_region_mask > 0]
        avg_b_channel = np.mean(b_channel_white_region_pixels) if b_channel_white_region_pixels.size > 0 else 0


        # L score (0–100)
        l_score = (np.mean(white_region_pixels) / 255) * 100

        # B score (less yellow = higher score)
        b_score = ((255 - avg_b_channel) / 255) * 100

        # Final whiteness
        whiteness = 0.7 * l_score + 0.3 * b_score

        # Calculate white percentage with adjustment
        total_area = cv2.countNonZero(core_mask)
        white_area = cv2.countNonZero(white_region_mask)
        white_percentage = ((white_area / total_area) * 100) if total_area > 0 else 0
        # Adjust and scale outputs
        # white_percentage += 10  # Adjustment factor
        # whiteness = (whiteness / 255) * 100  # Scale to 0-100
        # No scaling for b_channel, it's already in a useful range (-128 to 127)
        return white_percentage, whiteness, avg_b_channel
    except Exception as e:
        logging.error(f"Error in color analysis: {str(e)}")
        return 0, 0, 0

In [6]:
import numpy as np
# ===========================================================================
# Process a Single Image with Error Handling
# Purpose: Extract metrics and visualizations from one image
# ===========================================================================


def process_single_image(image, mm_per_pixel, thresholds={"min_contour_area": 100}):
    """
    Processes a single image to detect makhanas and compute their metrics.


    Args:
        image (numpy.ndarray): Input image in BGR format.
        mm_per_pixel (float): Calibration factor for pixel-to-mm conversion.
        thresholds (dict): Filtering thresholds (e.g., min_contour_area).


    Returns:
        tuple: (makhana_metrics, image_with_contours, segmentation_vis, individual_masks).
    """
    try:
        if image is None or image.size == 0:
            logging.error("Input image is None or empty")
            raise ValueError("Input image is None or empty")
        # --- Preprocessing ---
        image_with_contours = image.copy()  # Copy for contour overlay
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Grayscale conversion
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)  # Reduce noise
        # Adaptive thresholding with Otsu, adjusted for faint edges
        otsu_val, _ = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        adjusted_thresh = max(0, int(otsu_val - 15))  # Lower threshold for better detection
        _, binary = cv2.threshold(blurred, adjusted_thresh, 255, cv2.THRESH_BINARY)
        # Dilate to connect object parts
        kernel = np.ones((5, 5), np.uint8)
        dilated = cv2.dilate(binary, kernel, iterations=2)
        # Refine with distance transform
        dist = cv2.distanceTransform(dilated, cv2.DIST_L2, 5)
        _, refined = cv2.threshold(dist, 1.5, 255, 0)
        refined = refined.astype(np.uint8)
        # Segment objects via connected components
        num_labels, markers = cv2.connectedComponents(refined)
        unique_labels = list(range(1, num_labels))  # Exclude background
        # --- Metric Extraction ---
        makhana_metrics = []
        individual_masks = []
        detected_objects = []      # Step 1 Change
        for label in unique_labels:
            mask = (markers == label).astype(np.uint8) * 255
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not contours:
                continue
            cnt = max(contours, key=cv2.contourArea)

            points = cnt[:, 0, :]      # Convert contour to Nx2 array

            max_dist = 0

            feret_pt1 = None
            feret_pt2 = None

            for i in range(len(points)):
                for j in range(i + 1, len(points)):
                    d = np.linalg.norm(points[i] - points[j])

                    if d > max_dist:
                        max_dist = d
                        feret_pt1 = tuple(points[i])
                        feret_pt2 = tuple(points[j])

            max_feret_mm = max_dist * mm_per_pixel

            M = cv2.moments(cnt)

            if M["m00"] == 0:
                continue

            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            area_pixels = cv2.contourArea(cnt)
            if area_pixels < thresholds['min_contour_area']:
                continue
            # Smooth contour for better perimeter measurement
            epsilon = 0.005 * cv2.arcLength(cnt, True)
            cnt_smooth = cv2.approxPolyDP(cnt, epsilon, True)
            perimeter_pixels = cv2.arcLength(cnt_smooth, True)
            # Calculate circularity with enhancement
            circularity = min(1.0, (4 * math.pi * area_pixels) / (perimeter_pixels ** 2)) if perimeter_pixels > 0 else 0
            #circularity = circularity ** 3  # Amplify differences
            # Compute aspect ratio using fitted ellipse
            if len(cnt) >= 5:
                ellipse = cv2.fitEllipse(cnt)
                major_axis, minor_axis = max(ellipse[1]), min(ellipse[1])
                aspect_ratio = minor_axis / major_axis if major_axis > 0 else 0
            else:
                aspect_ratio = 0

            # Perform color analysis
            full_mask = np.zeros_like(gray)
            cv2.drawContours(full_mask, [cnt], -1, 255, -1)
            white_percentage, whiteness, avg_b_channel = analyze_color(image, full_mask)
            # Compile metrics
            metrics = {
                "Original Label": label, # OPEN CV LABEL [Step 6 Change]
                "Label": label, #Will later become row wise label
                "Effective Diameter (mm)": math.sqrt((4 * area_pixels) / math.pi) * mm_per_pixel,
                "Circularity": circularity,
                "Aspect Ratio": aspect_ratio,
                "Max Feret Diameter (mm)": max_feret_mm,
                "White Percentage": white_percentage,
                "Whiteness": whiteness,
                "Average B-Channel": avg_b_channel
            }
            metrics["Grade"] = determine_grade(metrics)
            # Extract individual makhana image and mask
            x, y, w, h = cv2.boundingRect(cnt)
            padding = 10
            x_start = max(0, x - padding)
            y_start = max(0, y - padding)
            x_end = min(image.shape[1], x + w + padding)
            y_end = min(image.shape[0], y + h + padding)
            individual_image = image[y_start:y_end, x_start:x_end].copy()
            individual_mask = np.zeros((y_end - y_start, x_end - x_start), dtype=np.uint8)
            shifted_contour = cnt.copy()
            shifted_contour[:, :, 0] -= x_start
            shifted_contour[:, :, 1] -= y_start
            cv2.drawContours(individual_mask, [shifted_contour], -1, 255, -1)

            #STEP 2 Chage
            detected_objects.append({
              "cx": cx,
              "cy": cy,
              "image": individual_image,
              "mask": individual_mask,
              "metrics": metrics
          })


       # STEP 3: Ask for number of rows
        num_rows = int(input("Enter number of rows: "))

        # STEP 4: Cluster objects into rows using KMeans on cy
        y_values = np.array([[obj["cy"]] for obj in detected_objects])

        kmeans = KMeans(n_clusters=num_rows, random_state=42, n_init=10)
        kmeans.fit(y_values)

        for obj, row_id in zip(detected_objects, kmeans.labels_):
            obj["row"] = row_id

        # Re-rank cluster IDs so row 0 = topmost row
        cluster_centers = kmeans.cluster_centers_.flatten()
        sorted_cluster_order = np.argsort(cluster_centers)
        cluster_id_to_row_number = {
            old_id: new_row for new_row, old_id in enumerate(sorted_cluster_order)
        }
        for obj in detected_objects:
            obj["row"] = cluster_id_to_row_number[obj["row"]]

        # STEP 5: Create rows explicitly
        rows = [[] for _ in range(num_rows)]

        for obj in detected_objects:
            rows[obj["row"]].append(obj)

        # Sort each row from left to right
        for row in rows:
            row.sort(key=lambda obj: obj["cx"])
        print("Rows found:", len(rows))
        for r in rows:
            print([obj["cy"] for obj in r])
        # STEP 5 Change
        label = 1

        for row_idx, row in enumerate(rows):
            for obj in row:
                obj["metrics"]["Label"] = label
                obj["metrics"]["Row"] = row_idx

                makhana_metrics.append(obj["metrics"])

                individual_masks.append({
                    "image": obj["image"],
                    "mask": obj["mask"],
                    "metrics": obj["metrics"]
                })

                label += 1
        # print("Final Labels:")

        # for row in rows:
        #     print([obj["metrics"]["Label"] for obj in row])
        # print("\nRows:")
        # for row in rows:
        #     print([(obj["metrics"]["Label"], obj["cx"], obj["cy"]) for obj in row])

        # --- Visualization ---
        grade_colors = {
            "Grade A": (0, 255, 0),   # Green
            "Grade B": (0, 255, 255), # Cyan
            "Grade C": (0, 165, 255), # Orange-ish
            "Grade D": (0, 0, 255)    # Red
        }
        # STEP 7 Change
        label_to_grade = {m["Original Label"]: m["Grade"] for m in makhana_metrics}
        segmentation_vis = np.zeros_like(image)
        for label in unique_labels:
            grade = label_to_grade.get(label, "Grade D")
            color = grade_colors.get(grade, (255, 255, 255))  # Default white
            segmentation_vis[markers == label] = color
        logging.info(f"Processed image: detected {len(makhana_metrics)} makhanas")
        return makhana_metrics, image_with_contours, segmentation_vis, individual_masks
    except Exception as e:
        logging.error(f"Error processing image: {str(e)}")
        blank_image = np.zeros_like(image) if image is not None else np.zeros((100, 100, 3), dtype=np.uint8)
        return [], blank_image, blank_image, []

In [7]:
# ===========================================================================
# Batch Processing with Memory Management
# Purpose: Efficiently process multiple images in parallel
# ===========================================================================


def process_batch_images(image_paths, mm_per_pixel, batch_size=5, thresholds=THRESHOLDS):
    """
    Processes a batch of images in parallel with memory-efficient batching.


    Args:
        image_paths (list): List of image file paths.
        mm_per_pixel (float): Calibration factor.
        batch_size (int): Number of images per processing batch (default: 5).
        thresholds (dict): Processing thresholds.


    Returns:
        tuple: (all_metrics, all_image_with_contours, all_segmentation_vis, all_individual_masks).
    """
    all_metrics = []
    all_image_with_contours = []
    all_segmentation_vis = []
    all_individual_masks = []
    # Process images in batches to manage memory
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i + batch_size]
        images = [cv2.imread(path) for path in batch_paths]
        with ProcessPoolExecutor() as executor:
            futures = [executor.submit(process_single_image, img, mm_per_pixel, thresholds) for img in images]
            for future in as_completed(futures):
                try:
                    metrics, contours, seg_vis, masks = future.result()
                    all_metrics.append(metrics)
                    all_image_with_contours.append(contours)
                    all_segmentation_vis.append(seg_vis)
                    all_individual_masks.append(masks)
                except Exception as e:
                    logging.error(f"Error in batch processing: {str(e)}")
                    all_metrics.append([])
                    all_image_with_contours.append(np.zeros((100, 100, 3), dtype=np.uint8))
                    all_segmentation_vis.append(np.zeros((100, 100, 3), dtype=np.uint8))
                    all_individual_masks.append([])
    return all_metrics, all_image_with_contours, all_segmentation_vis, all_individual_masks



In [8]:
# ===========================================================================
# Caching Intermediate Results
# Purpose: Optimize performance by reusing computed metrics
# ===========================================================================


def convert_to_serializable(obj):
    """
    Converts objects to JSON-serializable formats.


    Args:
        obj: Object to convert (e.g., NumPy types, arrays, dicts).


    Returns:
        JSON-compatible version of the object.
    """
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    return obj


def save_metrics(cache_dir, sample_num, metrics):
    """
    Saves metrics to a JSON file in the cache directory.


    Args:
        cache_dir (str): Directory for storing cached files.
        sample_num (int): Sample identifier for file naming.
        metrics (list): Metrics to cache.
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"sample_{sample_num}.json")
    try:
        serializable_metrics = convert_to_serializable(metrics)
        with open(cache_file, 'w') as f:
            json.dump(serializable_metrics, f, indent=2)
        logging.info(f"Cached metrics for sample {sample_num} to {cache_file}")
    except Exception as e:
        logging.error(f"Failed to save metrics for sample {sample_num}: {str(e)}")


def load_metrics(cache_dir, sample_num):
    """
    Loads cached metrics from a file if available.


    Args:
        cache_dir (str): Directory containing cached files.
        sample_num (int): Sample identifier.


    Returns:
        list or None: Cached metrics or None if unavailable.
    """
    cache_file = os.path.join(cache_dir, f"sample_{sample_num}.json")
    if os.path.exists(cache_file):
        try:
            with open(cache_file, 'r') as f:
                metrics = json.load(f)
            logging.info(f"Loaded cached metrics for sample {sample_num} from {cache_file}")
            return metrics
        except Exception as e:
            logging.error(f"Failed to load cache for sample {sample_num}: {str(e)}")
            return None
    return None


In [9]:
# ===========================================================================
# Batch Report Generation with Error Handling
# Purpose: Create a detailed PDF report summarizing analysis results
# ===========================================================================


def generate_batch_report(all_metrics, sample_images, segmentation_images, individual_masks_per_sample, mm_per_pixel, thresholds, num_samples, calib_data=None, report_metadata=None):
    """
    Generates a multi-page PDF report with analysis results and visualizations.


    Args:
        all_metrics (list): Metrics for all makhanas across samples.
        sample_images (list): Images with contours.
        segmentation_images (list): Segmentation visualizations.
        individual_masks_per_sample (list): Individual makhana data per sample.
        mm_per_pixel (float): Calibration factor.
        thresholds (dict): Processing thresholds.
        num_samples (int): Number of samples analyzed.
    """
    try:
        def add_watermark_and_footer(fig):
            """Adds a watermark and footer to each report page."""
            fig.text(0.5, 0.5, "CSIR-CSIO", fontsize=50, color='gray', ha='center', va='center', alpha=0.5, rotation=45)
            footer_line1 = "Report generated by Automated Makhana Grading System developed by CSIR-CSIO"
            if report_metadata:
                footer_line2 = f"v{report_metadata.get('Software Version', 'N/A')} | Operator: {report_metadata.get('Operator', 'N/A')} | {report_metadata.get('Date', '')} {report_metadata.get('Time', '')}"
                fig.text(0.5, 0.015, footer_line1, fontsize=8, ha='center', va='bottom')
                fig.text(0.5, 0.002, footer_line2, fontsize=6, ha='center', va='bottom', color='gray')
            else:
                fig.text(0.5, 0.01, footer_line1, fontsize=8, ha='center', va='bottom')


        # Formatting specifications for metrics display
        format_dict = {
            "Effective Diameter (mm)": ".2f",
            "Circularity": ".3f",
            "Aspect Ratio": ".3f",
            "White Percentage": ".1f",
            "Whiteness": ".1f"
        }
        param_to_threshold = {
            "Effective Diameter (mm)": "min_diameter",
            "Circularity": "min_circularity",
            "Aspect Ratio": "min_aspect_ratio",
            "White Percentage": "min_white_percentage",
            "Whiteness": "min_whiteness"
        }


        report_name = f"Makhana_Batch_Report_{get_ist_now().strftime('%Y%m%d_%H%M')}.pdf"
        with PdfPages(report_name) as pdf:
            # --- Front Page Summary ---
            plt.figure(figsize=(11, 8.5))
            if report_metadata:
                meta_lines = [f"{k}: {v}" for k, v in report_metadata.items()]
                meta_text = "  |  ".join(meta_lines[:4]) + "\n" + "  |  ".join(meta_lines[4:])
                plt.text(0.5, 0.75, meta_text, fontsize=8, ha='center', color='dimgray')
            plt.text(0.5, 0.9, "Makhana Batch Report", fontsize=20, ha='center')
            plt.text(0.5, 0.8, f"Generated on: {get_ist_now().strftime('%Y-%m-%d %H:%M')}", fontsize=12, ha='center')
            plt.text(0.5, 0.7, f"Total Samples: {num_samples}", fontsize=12, ha='center')
            total_makhanas = len(all_metrics)
            plt.text(0.5, 0.65, f"Total Makhanas Analyzed: {total_makhanas}", fontsize=12, ha='center')
            grade_dist = calculate_grade_distribution(all_metrics)
            grade_summary = "\n".join([f"{grade}: {d['count']} ({d['percentage']:.1f}%)" for grade, d in grade_dist.items()])
            plt.text(0.5, 0.6, "Grade Distribution:\n" + grade_summary, fontsize=12, ha='center', va='top')
            plt.axis('off')
            if all_metrics:
                avg_diameter = np.mean([m['Effective Diameter (mm)'] for m in all_metrics if 'Effective Diameter (mm)' in m and not np.isnan(m['Effective Diameter (mm)'])])
                avg_circularity = np.mean([m['Circularity'] for m in all_metrics if 'Circularity' in m and not np.isnan(m['Circularity'])])
                avg_aspect_ratio = np.mean([m['Aspect Ratio'] for m in all_metrics if 'Aspect Ratio' in m and not np.isnan(m['Aspect Ratio'])])
                avg_white_percentage = np.mean([m['White Percentage'] for m in all_metrics if 'White Percentage' in m and not np.isnan(m['White Percentage'])])
                avg_whiteness = np.mean([m['Whiteness'] for m in all_metrics if 'Whiteness' in m and not np.isnan(m['Whiteness'])])
            else:
                avg_diameter = avg_circularity = avg_aspect_ratio = avg_white_percentage = avg_whiteness = "N/A"
            summary_data = [
                ["Avg Diameter (mm)", f"{avg_diameter:.2f}" if isinstance(avg_diameter, (int, float)) else avg_diameter],
                ["Avg Circularity", f"{avg_circularity:.2f}" if isinstance(avg_circularity, (int, float)) else avg_circularity],
                ["Avg Aspect Ratio", f"{avg_aspect_ratio:.2f}" if isinstance(avg_aspect_ratio, (int, float)) else avg_aspect_ratio],
                ["Avg White %", f"{avg_white_percentage:.1f}%" if isinstance(avg_white_percentage, (int, float)) else avg_white_percentage],
                ["Avg Whiteness (L*)", f"{avg_whiteness:.1f}" if isinstance(avg_whiteness, (int, float)) else avg_whiteness]
            ]
            plt.table(cellText=summary_data, colLabels=["Parameter", "Value"], cellLoc='center', bbox=[0.3, 0.2, 0.4, 0.2])
            plt.figtext(0.1, 0.1, f"Calibration: {mm_per_pixel:.4f} mm/pixel\nDate: {get_ist_now().strftime('%Y-%m-%d %H:%M')}", fontsize=7)
            add_watermark_and_footer(plt.gcf())
            pdf.savefig()
            plt.close()

            # --- Calibration Information Page ---
            fig = plt.figure(figsize=(11, 8.5))
            fig.text(0.5, 0.94, "Calibration Information", fontsize=20, ha='center', fontweight='bold')

            if calib_data:
                method_display = {
                    'reference': 'Reference Object',
                    'geometric': 'Geometric (Camera Specs)',
                    'manual': 'Manual Entry'
                }.get(calib_data.get('method', ''), 'N/A')

                calib_fields = [
                    ("Calibration Method", method_display),
                    ("Reference Object", calib_data.get('reference_object', 'N/A')),
                    ("Actual Diameter", f"{calib_data.get('actual_diameter_mm', 'N/A')} mm"),
                    ("Measured Diameter", f"{calib_data.get('measured_diameter_px', 'N/A')} pixels"),
                    ("Pixel Resolution", f"{mm_per_pixel:.4f} mm/pixel"),
                    ("Calibration Status", calib_data.get('calibration_status', 'Successful')),
                    ("Calibration Date", calib_data.get('calibration_date', get_ist_now().strftime('%Y-%m-%d'))),
                    ("Image Resolution", calib_data.get('image_resolution', 'N/A')),
                    ("Camera", calib_data.get('camera_model', 'N/A')),
                ]
            else:
                calib_fields = [("Calibration Info", "Not Available")]

            # Text on the left half
            text_ax = fig.add_axes([0.05, 0.15, 0.45, 0.7])
            text_ax.axis('off')
            y_pos = 0.95
            for label, value in calib_fields:
                text_ax.text(0.55, y_pos, f"{label} :", fontsize=12, ha='right', fontweight='bold', transform=text_ax.transAxes)
                text_ax.text(0.6, y_pos, str(value), fontsize=12, ha='left', transform=text_ax.transAxes)
                y_pos -= 0.09

            # Coin image on the right half
            img_ax = fig.add_axes([0.55, 0.15, 0.4, 0.65])
            ref_image_path = calib_data.get('reference_image_path') if calib_data else None
            if ref_image_path and os.path.exists(ref_image_path):
                ref_img = cv2.imread(ref_image_path)
                img_ax.imshow(cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB))
                img_ax.set_title("Reference Object Image", fontsize=11)
            else:
                img_ax.text(0.5, 0.5, "Image Not Available", ha='center', va='center')
            img_ax.axis('off')

            add_watermark_and_footer(fig)
            pdf.savefig(fig)
            plt.close()


            # --- Sample Analyses ---
            for sample_num in range(1, num_samples + 1):
                sample_data = [m for m in all_metrics if m.get('Sample') == sample_num]
                plt.figure(figsize=(11, 8.5))
                plt.suptitle(f"Sample {sample_num} - Segmentation", fontsize=16)
                plt.subplot(1, 2, 1)
                if sample_images[sample_num-1] is not None:
                    plt.imshow(cv2.cvtColor(sample_images[sample_num-1], cv2.COLOR_BGR2RGB))
                    plt.title("Original Image")
                else:
                    plt.text(0.5, 0.5, "Image Not Available", ha='center', va='center')
                    plt.title("Original Image (Failed)")
                plt.axis('off')
                plt.subplot(1, 2, 2)
                if segmentation_images[sample_num-1] is not None:
                    plt.imshow(cv2.cvtColor(segmentation_images[sample_num-1], cv2.COLOR_BGR2RGB))
                    plt.title("Segmented Image")
                else:
                    plt.text(0.5, 0.5, "Image Not Available", ha='center', va='center')
                    plt.title("Segmented Image (Failed)")
                plt.axis('off')
                add_watermark_and_footer(plt.gcf())
                pdf.savefig()
                plt.close()


                individual_masks = individual_masks_per_sample[sample_num-1]
                makhanas_per_page = 4
                total_pages = math.ceil(len(individual_masks) / makhanas_per_page)
                for page in range(total_pages):
                    plt.figure(figsize=(11, 8.5))
                    plt.suptitle(f"Sample {sample_num} - Makhanas (Page {page+1}/{total_pages})", fontsize=16)
                    start_idx = page * makhanas_per_page
                    end_idx = min((page + 1) * makhanas_per_page, len(individual_masks))
                    for i, mask_data in enumerate(individual_masks[start_idx:end_idx]):
                        metrics = mask_data["metrics"]
                        plt.subplot(2, 2, i+1)
                        img = mask_data["image"]
                        mask = mask_data["mask"]
                        highlighted = img.copy()
                        for c in range(3):
                            highlighted[:,:,c] = cv2.bitwise_and(highlighted[:,:,c], mask)
                        grade = metrics["Grade"]
                        border_colors = {"Grade A": (0, 255, 0), "Grade B": (0, 255, 255), "Grade C": (0, 165, 255), "Grade D": (0, 0, 255)}
                        bordered = cv2.copyMakeBorder(highlighted, 3, 3, 3, 3, cv2.BORDER_CONSTANT, value=border_colors.get(grade, (255, 255, 255)))
                        plt.imshow(cv2.cvtColor(bordered, cv2.COLOR_BGR2RGB))
                        plt.title(f"Makhana #{metrics['Label']} - {grade}", fontsize=10)
                        metrics_lines = []
                        for param, fmt in zip(
                            ["Effective Diameter (mm)", "Circularity", "Aspect Ratio", "White Percentage", "Whiteness"],
                            [".2f", ".3f", ".3f", ".1f", ".1f"]
                        ):
                            value = metrics[param]
                            line = f"{param}: {value:{fmt}}"
                            if grade in COMPARISON_GRADE:
                                comparison_grade = COMPARISON_GRADE[grade]
                                threshold_key = param_to_threshold[param]
                                threshold = BATCH_GRADES[comparison_grade][threshold_key]
                                if value < threshold:
                                    line += " (*)"
                            metrics_lines.append(line)
                        metrics_text = "\n".join(metrics_lines)
                        plt.text(0.5, -0.05, metrics_text, fontsize=7, ha='center', va='top', transform=plt.gca().transAxes, color='black')
                        plt.axis('off')
                    plt.subplots_adjust(bottom=0.3, hspace=0.5)
                    plt.figtext(0.1, 0.05, "(*): Metric does not meet the threshold for the next higher grade.", fontsize=8, ha='left')
                    add_watermark_and_footer(plt.gcf())
                    pdf.savefig()
                    plt.close()


                fig = plt.figure(figsize=(11, 8.5))
                gs = GridSpec(2, 2, figure=fig, height_ratios=[3, 1])
                ax_table = fig.add_subplot(gs[0, :])
                ax_pie1 = fig.add_subplot(gs[1, 0])
                ax_pie2 = fig.add_subplot(gs[1, 1])
                params = ["Effective Diameter (mm)", "Circularity", "Aspect Ratio", "White Percentage", "Whiteness"]
                stats = [["Parameter", "Min", "Max", "Mean", "Median", "Std"]]
                for param in params:
                    values = [m[param] for m in sample_data if param in m and not np.isnan(m[param])]
                    if values:
                        min_val = min(values)
                        max_val = max(values)
                        mean_val = np.mean(values)
                        median_val = np.percentile(values, 50)
                        std_val = np.std(values)
                        fmt = format_dict[param]
                        stats.append([param] + [f"{stat:{fmt}}" for stat in [min_val, max_val, mean_val, median_val, std_val]])
                if len(stats) > 1:
                    ax_table.table(cellText=stats, colLabels=stats[0], loc="center", cellLoc="center")
                ax_table.axis('off')
                grade_counts = Counter(m['Grade'] for m in sample_data if 'Grade' in m)
                total_makhanas = len(sample_data)
                grade_order = ['Grade A', 'Grade B', 'Grade C', 'Grade D']
                labels = [grade for grade in grade_order if grade in grade_counts]
                sizes_percent = [grade_counts[grade] / total_makhanas * 100 for grade in labels] if total_makhanas > 0 else []
                sizes_count = [grade_counts[grade] for grade in labels]
                if sizes_percent:
                    ax_pie1.pie(sizes_percent, labels=labels, autopct='%1.1f%%', startangle=90)
                    ax_pie1.set_title("Grade Distribution by Percentage")
                else:
                    ax_pie1.text(0.5, 0.5, "No Data", ha='center', va='center')
                if sizes_count:
                    ax_pie2.pie(sizes_count, labels=labels, autopct=lambda p: '{:.0f}'.format((p / 100) * total_makhanas), startangle=90)
                    ax_pie2.set_title("Grade Distribution by Count")
                else:
                    ax_pie2.text(0.5, 0.5, "No Data", ha='center', va='center')
                fig.suptitle(f"Sample {sample_num} - Summary", fontsize=16)
                add_watermark_and_footer(fig)
                pdf.savefig(fig)
                plt.close(fig)


            # --- Batch Summary ---
            plt.figure(figsize=(11, 8.5))
            plt.suptitle("Batch Summary", fontsize=16)
            if all_metrics:
                batch_stats = [["Parameter", "Min", "Max", "Mean", "Median", "Std"]]
                params = ["Effective Diameter (mm)", "Circularity", "Aspect Ratio", "White Percentage", "Whiteness"]
                for param in params:
                    values = [m[param] for m in all_metrics if param in m and not np.isnan(m[param])]
                    if values:
                        min_val = min(values)
                        max_val = max(values)
                        mean_val = np.mean(values)
                        median_val = np.percentile(values, 50)
                        std_val = np.std(values)
                        fmt = format_dict[param]
                        batch_stats.append([param] + [f"{stat:{fmt}}" for stat in [min_val, max_val, mean_val, median_val, std_val]])
                if len(batch_stats) > 1:
                    plt.table(cellText=batch_stats, colLabels=batch_stats[0], loc="center", cellLoc="center")
            else:
                plt.text(0.5, 0.5, "No data for batch statistics", ha='center', va='center')
            plt.axis('off')
            add_watermark_and_footer(plt.gcf())
            pdf.savefig()
            plt.close()


            # --- Batch Grade Distribution ---
            batch_grade_dist = calculate_grade_distribution(all_metrics)
            plt.figure(figsize=(11, 8.5))
            plt.suptitle("Batch Grade Distribution", fontsize=16)
            grades = list(batch_grade_dist.keys())
            percentages = [d['percentage'] for d in batch_grade_dist.values()]
            colors = ['gold', 'yellowgreen', 'lightcoral', 'lightskyblue']
            plt.subplot(2, 1, 1)
            if percentages:
                plt.pie(percentages, labels=grades, colors=colors, autopct='%1.1f%%', startangle=140)
            else:
                plt.text(0.5, 0.5, "No Data", ha='center', va='center')
            cell_text = [[d['count'], f"{d['percentage']:.1f}%"] for d in batch_grade_dist.values()]
            plt.subplot(2, 1, 2)
            if cell_text:
                plt.table(cellText=cell_text, rowLabels=grades, colLabels=["Count", "%"], loc="center", cellLoc="center")
            plt.axis('off')
            plt.tight_layout(rect=[0, 0, 1, 0.95])
            add_watermark_and_footer(plt.gcf())
            pdf.savefig()
            plt.close()


            # --- Histograms for Key Metrics (Page 1) ---
            plt.figure(figsize=(11, 8.5))
            plt.suptitle("Distribution of Key Metrics - Page 1", fontsize=16)
            metrics_to_plot = ["Effective Diameter (mm)", "Circularity", "Whiteness"]
            any_data = False
            for i, metric in enumerate(metrics_to_plot, 1):
                values = [m[metric] for m in all_metrics if metric in m and not np.isnan(m[metric])]
                plt.subplot(3, 1, i)
                if values and len(values) > 0:
                    plt.hist(values, bins=20, color='blue', alpha=0.7)
                    plt.title(metric)
                    plt.xlabel(metric)
                    plt.ylabel("Frequency")
                    any_data = True
                else:
                    plt.text(0.5, 0.5, "No Data", ha='center', va='center')
                    plt.axis('off')
            if any_data:
                plt.tight_layout(rect=[0, 0, 1, 0.95])
                add_watermark_and_footer(plt.gcf())
                pdf.savefig()
            else:
                plt.text(0.5, 0.5, "No data available for histograms - Page 1", ha='center', va='center', transform=plt.gcf().transFigure)
                add_watermark_and_footer(plt.gcf())
                pdf.savefig()
            plt.close()


            # --- Histograms for Key Metrics (Page 2) ---
            plt.figure(figsize=(11, 8.5))
            plt.suptitle("Distribution of Key Metrics - Page 2", fontsize=16)
            metrics_to_plot = ["White Percentage", "Aspect Ratio"]
            any_data = False
            colors = ['purple', 'orange']
            for i, (metric, color) in enumerate(zip(metrics_to_plot, colors), 1):
                values = [m[metric] for m in all_metrics if metric in m and not np.isnan(m[metric])]
                plt.subplot(2, 1, i)
                if values and len(values) > 0:
                    plt.hist(values, bins=20, color=color, alpha=0.7)
                    plt.title(metric)
                    plt.xlabel(metric)
                    plt.ylabel("Frequency")
                    any_data = True
                else:
                    plt.text(0.5, 0.5, "No Data", ha='center', va='center')
                    plt.axis('off')
            if any_data:
                plt.tight_layout(rect=[0, 0, 1, 0.95])
                add_watermark_and_footer(plt.gcf())
                pdf.savefig()
            else:
                plt.text(0.5, 0.5, "No data available for histograms - Page 2", ha='center', va='center', transform=plt.gcf().transFigure)
                add_watermark_and_footer(plt.gcf())
                pdf.savefig()
            plt.close()


            # --- Batch Grade-Specific Stats ---
            plt.figure(figsize=(11, 8.5))
            plt.suptitle("Batch Grade-Specific Stats", fontsize=16)
            grade_stats = [["Grade"] + params]
            for grade in ["Grade A", "Grade B", "Grade C", "Grade D"]:
                grade_data = [m for m in all_metrics if m.get("Grade") == grade]
                if grade_data:
                    stats_row = [grade]
                    for param in params:
                        values = [m[param] for m in grade_data if param in m and not np.isnan(m[param])]
                        avg = np.mean(values) if values else "N/A"
                        stats_row.append(f"{avg:.2f}" if isinstance(avg, (int, float)) else avg)
                    grade_stats.append(stats_row)
            if len(grade_stats) > 1:
                plt.table(cellText=grade_stats, colLabels=grade_stats[0], loc="center", cellLoc="center")
            else:
                plt.text(0.5, 0.5, "No data for grade-specific stats", ha='center', va='center')
            plt.axis('off')
            add_watermark_and_footer(plt.gcf())
            pdf.savefig()
            plt.close()


            # --- Correlation Heatmap ---
            plt.figure(figsize=(11, 8.5))
            plt.suptitle("Correlation of Metrics", fontsize=16)
            params = ["Effective Diameter (mm)", "Circularity", "Aspect Ratio", "White Percentage", "Whiteness"]
            data_matrix = np.array([[m[p] for p in params if p in m and not np.isnan(m[p])] for m in all_metrics if all(p in m and not np.isnan(m[p]) for p in params)])
            if data_matrix.size > 0:
                corr_matrix = np.corrcoef(data_matrix.T)
                plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
                plt.colorbar()
                plt.xticks(range(len(params)), params, rotation=45, ha='right')
                plt.yticks(range(len(params)), params)
                for i in range(len(params)):
                    for j in range(len(params)):
                        plt.text(j, i, f"{corr_matrix[i, j]:.2f}", ha='center', va='center', color='black')
            else:
                plt.text(0.5, 0.5, "No Data for Correlation", ha='center', va='center')
            add_watermark_and_footer(plt.gcf())
            pdf.savefig()
            plt.close()


            # --- Batch Grade ---
            grade_points = {'Grade A': 100, 'Grade B': 66.666, 'Grade C': 33.333, 'Grade D': 0}
            total_points = sum(grade_points.get(m.get('Grade', 'Grade D'), 0) for m in all_metrics)
            average_score = total_points / len(all_metrics) if len(all_metrics) > 0 else 0
            if average_score >= 80:
                batch_grade = 'Grade A'
            elif average_score >= 60:
                batch_grade = 'Grade B'
            elif average_score >= 40:
                batch_grade = 'Grade C'
            else:
                batch_grade = 'Grade D'
            grade_colors = {'Grade A': 'green', 'Grade B': 'yellowgreen', 'Grade C': 'orange', 'Grade D': 'red'}
            plt.figure(figsize=(11, 8.5))
            plt.text(0.5, 0.9, 'Overall Batch Grade', fontsize=20, ha='center', va='center')
            plt.text(0.5, 0.7, f'Batch Grade: {batch_grade}', fontsize=24, ha='center', va='center', color=grade_colors[batch_grade], fontweight='bold')
            plt.text(0.5, 0.5, f'Average Score: {average_score:.2f}', fontsize=18, ha='center', va='center')
            plt.text(0.5, 0.3, 'Scoring System:\nGrade A: 100\nGrade B: 66.67\nGrade C: 33.33\nGrade D: 0', fontsize=12, ha='center', va='center')
            plt.text(0.5, 0.1, 'Batch Grade Thresholds:\nGrade A: >=80\nGrade B: >=60\nGrade C: >=40\nGrade D: <40', fontsize=12, ha='center', va='center')
            plt.axis('off')
            add_watermark_and_footer(plt.gcf())
            pdf.savefig()
            plt.close()


            # --- Glossary Section ---
            fig = plt.figure(figsize=(11, 8.5))
            fig.text(0.5, 0.95, "Glossary", fontsize=16, fontweight='bold', ha='center', va='center')
            left_ax = fig.add_axes([0.05, 0.1, 0.45, 0.8])
            right_ax = fig.add_axes([0.55, 0.1, 0.45, 0.8])
            left_ax.text(0, 1, "Makhana:\nAlso known as fox nuts or lotus seeds, makhana comes from the Euryale ferox plant, "
                              "a species native to eastern Asia. It’s a versatile ingredient used in snacks (roasted with spices), "
                              "desserts (like kheer), and savory dishes (curries). Nutritionally, makhana offers 9g of protein per 100g, "
                              "significant fiber, and antioxidants like gallic acid, which combat oxidative stress. Its low glycemic index "
                              "makes it ideal for diabetic diets, and its gluten-free nature suits various dietary needs.",
                        ha='left', va='top', wrap=True, fontsize=10)
            right_ax.text(0, 1, "Grading System:\nA standardized process to evaluate makhana quality based on physical attributes like size, shape, "
                                "and color purity. This system benefits stakeholders: sellers can price accurately, consumers can select "
                                "Grade A products, and exporters can meet international standards (e.g., ISO 22000 for food safety). Our "
                                "approach leverages computer vision—using algorithms like OpenCV’s contour detection—to automate grading, "
                                "processing up to 10,000 seeds per hour with 95% accuracy compared to manual methods.\n\n"
                                "Metrics:\n• Diameter: Measured in millimeters (mm) using the formula √(4 * area / π), where area is derived from "
                                "pixel counts in processed images. Larger diameters (>15mm) often indicate mature, high-quality seeds.\n"
                                "• Circularity: A shape metric scored from 0 to 1, calculated as (4 * π * area) / (perimeter_pixels²). Values near 1 "
                                "denote near-perfect roundness, preferred for aesthetic appeal in Grade A grades.\n"
                                "• Whiteness: The percentage of surface area free from blemishes, computed via pixel intensity histograms. "
                                "Values >90% signify top purity, critical for markets like Japan where visual quality is paramount.",
                          ha='left', va='top', wrap=True, fontsize=10)
            left_ax.axis('off')
            right_ax.axis('off')
            add_watermark_and_footer(fig)
            pdf.savefig(fig)
            plt.close()


            # --- Methodology Section ---
            fig = plt.figure(figsize=(11, 8.5))
            fig.text(0.5, 0.95, "Methodology", fontsize=16, fontweight='bold', ha='center', va='center')
            left_ax = fig.add_axes([0.05, 0.1, 0.45, 0.8])
            right_ax = fig.add_axes([0.55, 0.1, 0.45, 0.8])
            left_ax.text(0, 1, "Image Processing:\nHigh-resolution cameras (e.g., 12MP) capture makhana images under controlled lighting (5000K LED). "
                              "Our software, built with Python and OpenCV, processes these images in steps:\n"
                              "  1. Thresholding: Applies Otsu’s method to binarize images, separating seeds from the background.\n"
                              "  2. Contour Detection: Uses edge detection (Canny algorithm) to outline each seed, enabling area and perimeter_pixels "
                              "measurements.\n"
                              "  3. Metric Calculation: Computes diameter, circularity, and whiteness using the formulas above.\n"
                              "This pipeline processes batches in real-time, achieving a throughput of 50 images per second on a standard GPU.",
                        ha='left', va='top', wrap=True, fontsize=10)
            right_ax.text(0, 1, "Grading Algorithm:\nThe algorithm integrates metrics into a composite score (0–1) using a weighted sum: "
                                "Score = w₁ * Diameter_norm + w₂ * Circularity + w₃ * Whiteness_norm, where weights (w₁, w₂, w₃) are adjustable "
                                "(e.g., 0.4, 0.3, 0.3). Normalized values ensure fairness across metrics. Grades are assigned as:\n"
                                "  - Grade A: ≥0.85 (export quality)\n  - Grade B: 0.65–0.85 (retail)\n  - Grade C: 0.45–0.65 (local markets)\n"
                                "  - Grade D: <0.45 (unsuitable). This flexibility allows customization for different market needs.\n\n"
                                "Calibration:\nAccuracy is maintained by calibrating with a reference object (e.g., a 20mm coin) placed in the frame. "
                                "This accounts for lens distortion, focal length variations, and lighting changes. The system calculates a "
                                "pixel-to-mm ratio, ensuring measurements remain consistent across setups—crucial for scaling from lab to industrial use. "
                                "Regular recalibration (e.g., weekly) mitigates drift, maintaining an error margin below 2%.",
                          ha='left', va='top', wrap=True, fontsize=10)
            left_ax.axis('off')
            right_ax.axis('off')
            add_watermark_and_footer(fig)
            pdf.savefig(fig)
            plt.close()


            # --- Educational Content Section ---
            fig = plt.figure(figsize=(11, 8.5))
            fig.text(0.5, 0.95, "All About Makhana", fontsize=16, fontweight='bold', ha='center', va='center')
            left_ax = fig.add_axes([0.05, 0.1, 0.45, 0.8])
            right_ax = fig.add_axes([0.55, 0.1, 0.45, 0.8])
            left_ax.text(0, 1, "General Information:\nMakhana, a staple in Indian and Chinese cuisine, is dubbed a superfood for its health benefits. "
                              "With 350 kcal per 100g, it’s low in fat (0.1g) but rich in magnesium, potassium, and phosphorus—key for heart "
                              "and bone health. Traditionally, it’s roasted with ghee and spices or used in Ayurvedic remedies for insomnia "
                              "and digestion. Its cultivation in Bihar, India, supports thousands of farmers, making it an economic and cultural gem.",
                        ha='left', va='top', wrap=True, fontsize=10)
            right_ax.text(0, 1, "Why Quality Matters:\nHigh-quality makhana—larger, rounder, and whiter—commands Grade A prices (up to $10/kg vs. $4/kg for lower grades). "
                                "For sellers, consistent grading builds trust and boosts profits. Consumers enjoy better taste and texture, while exporters "
                                "meet strict standards (e.g., EU’s maximum residue limits). Quality also affects shelf life—whiter seeds resist spoilage longer.\n\n"
                                "Tips for Everyone:\n• Storage: Store in airtight containers at <25°C to preserve crispness; humidity above 60% risks mold growth.\n"
                                "• Buying: Check for uniformity—mixed sizes or dark spots suggest poor sorting. Smell for freshness (stale makhana smells musty).\n"
                                "• Sellers: Use grading data on packaging (e.g., ‘Grade A, >15mm’) to attract buyers and justify pricing.\n"
                                "• Cooking: Soak low-grade makhana in water to soften for soups if roasting isn’t viable.",
                          ha='left', va='top', wrap=True, fontsize=10)
            left_ax.axis('off')
            right_ax.axis('off')
            add_watermark_and_footer(fig)
            pdf.savefig(fig)
            plt.close()


            # --- Visual Aids Section ---
            fig = plt.figure(figsize=(11, 8.5))
            fig.text(0.5, 0.95, "Visual Guide", fontsize=16, fontweight='bold', ha='center', va='center')
            left_ax = fig.add_axes([0.05, 0.1, 0.45, 0.8])
            right_ax = fig.add_axes([0.55, 0.1, 0.45, 0.8])
            left_ax.text(0, 1, "How Grading Works (Flowchart):\n1. Capture Image → 2. Process Image (Otsu Thresholding & Canny Edge Detection) → "
                              "3. Measure Metrics (Diameter, Circularity, Whiteness) → 4. Compute Weighted Score → 5. Assign Grade\n"
                              "(Note: In practice, this could be visualized with arrows and boxes using matplotlib’s plotting tools.)",
                        ha='left', va='top', wrap=True, fontsize=10)
            right_ax.text(0, 1, "Grade Thresholds:", fontsize=12, fontweight='bold', ha='left', va='top')
            table_data = [
                ["Grade", "Diameter (mm)", "Circularity", "Whiteness (%)"],
                ["Grade A", "> 15", "> 0.9", "> 90"],
                ["Grade B", "12–15", "0.8–0.9", "80–90"],
                ["Grade C", "9–12", "0.7–0.8", "70–80"],
                ["Grade D", "< 9", "< 0.7", "< 70"]
            ]
            table = right_ax.table(cellText=table_data, loc='center', cellLoc='center')
            table.auto_set_font_size(False)
            table.set_fontsize(10)
            left_ax.axis('off')
            right_ax.axis('off')
            add_watermark_and_footer(fig)
            pdf.savefig(fig)
            plt.close()


        logging.info(f"Batch report generated: {report_name}")
        files.download(report_name)
    except Exception as e:
        logging.error(f"Error generating batch report: {str(e)}")
        if 'pdf' in locals():
            pdf.close()


In [10]:
def generate_batch_excel(all_metrics, num_samples, mm_per_pixel, report_metadata=None):
    """
    Generates an Excel report with four sheets:
    1. 'Makhana Data'      - one row per makhana with all metrics
    2. 'Grade Summary'     - grade distribution counts and percentages
    3. 'Batch Summary'     - min/max/mean/median/std for each parameter
    4. 'Report Info'       - calibration, totals, and overall batch grade

    Args:
        all_metrics (list): Metrics for all makhanas across samples.
        num_samples (int): Number of samples analyzed.
        mm_per_pixel (float): Calibration factor used for this run.
    """
    try:
        excel_name = f"Makhana_Batch_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

        # --- Sheet 1: Makhana Data ---
        columns = [
            "Sample", "Row", "Label", "Grade",
            "Effective Diameter (mm)", "Circularity", "Aspect Ratio", "Max Feret Diameter (mm)",
            "White Percentage", "Whiteness"
        ]
        rows_for_df = [{col: m.get(col, "") for col in columns} for m in all_metrics]
        df_data = pd.DataFrame(rows_for_df, columns=columns)
        df_data.sort_values(by=["Sample", "Label"], inplace=True)

        # --- Sheet 2: Grade Summary ---
        grade_dist = calculate_grade_distribution(all_metrics)
        summary_rows = [
            {"Grade": grade, "Count": d["count"], "Percentage": round(d["percentage"], 1)}
            for grade, d in grade_dist.items()
        ]
        df_summary = pd.DataFrame(summary_rows, columns=["Grade", "Count", "Percentage"])

        # --- Sheet 3: Batch Summary Stats ---
        params = ["Effective Diameter (mm)", "Circularity", "Aspect Ratio", "Max Feret Diameter (mm)", "White Percentage", "Whiteness"]
        stats_rows = []
        for param in params:
            values = [m[param] for m in all_metrics if param in m and not np.isnan(m[param])]
            if values:
                stats_rows.append({
                    "Parameter": param,
                    "Min": round(min(values), 3),
                    "Max": round(max(values), 3),
                    "Mean": round(np.mean(values), 3),
                    "Median": round(np.percentile(values, 50), 3),
                    "Std": round(np.std(values), 3)
                })
        df_stats = pd.DataFrame(stats_rows, columns=["Parameter", "Min", "Max", "Mean", "Median", "Std"])

        # --- Sheet 4: Report Info ---
        grade_points = {'Grade A': 100, 'Grade B': 66.666, 'Grade C': 33.333, 'Grade D': 0}
        total_points = sum(grade_points.get(m.get('Grade', 'Grade D'), 0) for m in all_metrics)
        average_score = total_points / len(all_metrics) if len(all_metrics) > 0 else 0
        if average_score >= 80:
            batch_grade = 'Grade A'
        elif average_score >= 60:
            batch_grade = 'Grade B'
        elif average_score >= 40:
            batch_grade = 'Grade C'
        else:
            batch_grade = 'Grade D'

        info_rows = [
            {"Field": "Report Generated", "Value": get_ist_now().strftime('%Y-%m-%d %H:%M')},
            {"Field": "Total Samples", "Value": num_samples},
            {"Field": "Total Makhanas Analyzed", "Value": len(all_metrics)},
            {"Field": "Calibration (mm/pixel)", "Value": round(mm_per_pixel, 4)},
            {"Field": "Overall Batch Grade", "Value": batch_grade},
            {"Field": "Average Score", "Value": round(average_score, 2)},
        ]
        if report_metadata:
            for k, v in report_metadata.items():
                info_rows.append({"Field": k, "Value": v})
        df_info = pd.DataFrame(info_rows, columns=["Field", "Value"])

        # --- Write all sheets to one Excel file ---
        with pd.ExcelWriter(excel_name, engine="openpyxl") as writer:
            df_data.to_excel(writer, sheet_name="Makhana Data", index=False)
            df_summary.to_excel(writer, sheet_name="Grade Summary", index=False)
            df_stats.to_excel(writer, sheet_name="Batch Summary", index=False)
            df_info.to_excel(writer, sheet_name="Report Info", index=False)

        logging.info(f"Excel report generated: {excel_name}")
        files.download(excel_name)
    except Exception as e:
        logging.error(f"Error generating Excel report: {str(e)}")

In [11]:

# ===========================================================================
# Batch Analysis Function
# Purpose: Orchestrate the entire analysis workflow
# ===========================================================================


def analyze_batch():
    """
    Manages the complete batch analysis process from image upload to report generation.


    Ensures all images and visualizations are included in the final report.
    """
    logging.info("Starting batch analysis")
    operator_name = input("Enter operator name: ")
    try:
        num_samples = int(input("Enter the number of samples to analyze: "))
        if num_samples <= 0:
            raise ValueError("Number of samples must be positive")
    except ValueError as e:
        logging.error(f"Invalid number of samples: {str(e)}")
        raise ValueError("Please enter a valid positive integer for the number of samples")
    logging.info(f"User requested {num_samples} samples")
    print(f"\nPlease upload exactly {num_samples} sample images at once.")
    uploaded = files.upload()
    if len(uploaded) != num_samples:
        logging.error(f"Upload mismatch: expected {num_samples}, got {len(uploaded)}")
        raise ValueError(f"Expected {num_samples} images, but got {len(uploaded)}.")
    image_paths = [(i+1, filename) for i, filename in enumerate(uploaded.keys())]
    mm_per_pixel, calib_data = get_calibration_factor()

    method_display = {
        'reference': 'Reference Object',
        'geometric': 'Geometric (Camera Specs)',
        'manual': 'Manual Entry'
    }.get(calib_data.get('method', ''), 'N/A')

    now = get_ist_now()
    report_metadata = {
        "Software Version": SOFTWARE_VERSION,
        "Operator": operator_name,
        "Date": now.strftime("%Y-%m-%d"),
        "Time": now.strftime("%H:%M:%S"),
        "Camera": calib_data.get('camera_model', 'N/A'),
        "Image Resolution": calib_data.get('image_resolution', 'N/A'),
        "Calibration Method": method_display,
        "Calibration Constant": f"{mm_per_pixel:.4f} mm/pixel"
    }
    logging.info(f"Calibration set to {mm_per_pixel:.4f} mm/pixel")
    print(f"Calibration factor (mm_per_pixel): {mm_per_pixel:.6f}")
    # Unique cache directory for this run
    run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    CACHE_DIR = f"cache_{run_timestamp}"
    os.makedirs(CACHE_DIR, exist_ok=True)
    logging.info(f"Using unique cache directory: {CACHE_DIR}")
    # Initialize result storage
    sample_images = [None] * num_samples
    segmentation_images = [None] * num_samples
    individual_masks_per_sample = [[] for _ in range(num_samples)]
    batch_metrics = []


    def process_image(sample_num, image_path):
        """
        Processes a single image with caching support.


        Args:
            sample_num (int): Sample number.
            image_path (str): Path to the image.


        Returns:
            tuple: (sample_num, metrics, contours, seg_vis, masks).
        """
        cached_metrics = load_metrics(CACHE_DIR, sample_num)
        if cached_metrics:
            logging.info(f"Using cached metrics for sample {sample_num} within this run")
            return sample_num, cached_metrics, None, None, []
        try:
            image = cv2.imread(image_path)
            if image is None:
                raise FileNotFoundError(f"Could not load image {image_path}")
            metrics, contours, seg_vis, masks = process_single_image(image, mm_per_pixel, THRESHOLDS)
            for m in metrics:
                m['Sample'] = sample_num
            save_metrics(CACHE_DIR, sample_num, metrics)
            return sample_num, metrics, contours, seg_vis, masks
        except Exception as e:
            logging.error(f"Error processing sample {sample_num}: {str(e)}")
            return sample_num, [], None, None, []




    # Process images in parallel
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = {executor.submit(process_image, sn, path): sn for sn, path in image_paths}
        for future in concurrent.futures.as_completed(futures):
            sample_num = futures[future]
            try:
                sn, metrics, contours, seg_vis, masks = future.result()
                batch_metrics.extend(metrics)
                if contours is not None:
                    sample_images[sn-1] = contours
                    segmentation_images[sn-1] = seg_vis
                    individual_masks_per_sample[sn-1] = masks
                logging.info(f"Sample {sn}: {len(metrics)} makhanas analyzed")
            except Exception as e:
                logging.error(f"Error in processing sample {sample_num}: {str(e)}")


    generate_batch_report(batch_metrics, sample_images, segmentation_images, individual_masks_per_sample, mm_per_pixel, THRESHOLDS, num_samples, calib_data, report_metadata)
    generate_batch_excel(batch_metrics, num_samples, mm_per_pixel, report_metadata)
    logging.info("Batch analysis completed successfully!")


In [12]:
# ===========================================================================
# Main Execution
# ===========================================================================


if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    analyze_batch()

Enter operator name: end
Enter the number of samples to analyze: 1

Please upload exactly 1 sample images at once.


Saving grade a.tif to grade a.tif
1. Reference Object (Recommended)
2. Geometric (Camera Specs)
3. Manual Entry
Select method (1-3): 3
Enter camera model (e.g., JAI AD-130GE): jd
Enter known mm/pixel value: 0.1581
Calibration factor (mm_per_pixel): 0.158100
Enter number of rows: 4
Rows found: 4
[158, 95, 114, 94, 103]
[363, 334, 312, 328, 313]
[643, 593, 581, 561, 535]
[847, 841, 830, 844, 796]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>